In [58]:
import pandas as pd

# Load the dataset using Latin-1 encoding to handle special symbols like the British Pound sign
df = pd.read_csv(r"C:\Users\sanjay\OneDrive\Desktop\CS50\004.online_retail_dataset\OnlineRetail.csv", encoding='ISO-8859-1', low_memory=False)

In [59]:
print(df.shape)
print(df.columns)
print(df.info())
df.head(50)

(541909, 8)
Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 67.3 MB
None


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom


In [60]:
missing_count = df.isnull().sum()

missing_percent = (
    df.isnull().sum() / len(df)
) * 100

missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent
})

missing_report = missing_report.sort_values(
    "Missing %",
    ascending=False
)

print(missing_report)

             Missing Count  Missing %
CustomerID          135080  24.926694
Description           1454   0.268311
StockCode                0   0.000000
InvoiceNo                0   0.000000
Quantity                 0   0.000000
InvoiceDate              0   0.000000
UnitPrice                0   0.000000
Country                  0   0.000000


In [61]:
# Iterate through each column name specified in the list to profile them individually
for col in [
    'InvoiceNo', 
    'StockCode', 
    'Description', 
    'Quantity', 
    'InvoiceDate',
    'UnitPrice', 
    'CustomerID', 
    'Country'
]:

    # Calculate the frequency of unique values in the current column, including missing values (NaN),
    # and print only the top 20 most frequent entries to identify data patterns and anomalies
    print(df[col].value_counts(dropna=False).head(20))

# Calculate and return the grand total of rows that are 100% identical duplicates across all columns
df.duplicated().sum()

# Display the data type of every column to see how Pandas stored the data (e.g., object, int64, float64)
df.dtypes

InvoiceNo
573585    1114
581219     749
581492     731
580729     721
558475     705
579777     687
581217     676
537434     675
580730     662
538071     652
580367     650
580115     645
581439     635
580983     629
578344     622
538349     620
578347     606
537638     601
537237     597
536876     593
Name: count, dtype: int64
StockCode
85123A    2313
22423     2203
85099B    2159
47566     1727
20725     1639
84879     1502
22720     1477
22197     1476
21212     1385
20727     1350
22383     1348
22457     1280
23203     1267
POST      1256
22386     1251
22469     1239
22960     1229
21931     1214
22086     1210
22411     1202
Name: count, dtype: int64
Description
WHITE HANGING HEART T-LIGHT HOLDER    2369
REGENCY CAKESTAND 3 TIER              2200
JUMBO BAG RED RETROSPOT               2159
PARTY BUNTING                         1727
LUNCH BAG RED RETROSPOT               1638
ASSORTED COLOUR BIRD ORNAMENT         1501
SET OF 3 CAKE TINS PANTRY DESIGN      1473
NaN            

InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object

In [62]:
# Strip leading and trailing whitespaces from column names to prevent indexing errors
df.columns = df.columns.str.strip()

# Remove exact duplicate rows from the DataFrame to ensure transactional integrity
df = df.drop_duplicates()

# Filter out transactions that represent cancellations or data entry anomalies
# This removes rows where Quantity or UnitPrice is less than or equal to zero
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

# Convert the InvoiceDate column from a string object to a standardized datetime format
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format="%m/%d/%Y %H:%M")

# Fill missing CustomerID values with a placeholder integer before converting data types
df["CustomerID"] = df["CustomerID"].fillna(0)

# Convert CustomerID from a floating-point number to an integer to remove the trailing decimal
df["CustomerID"] = df["CustomerID"].astype(int)

# Convert CustomerID to a string type, replacing the placeholder '0' with an explicit 'Missing' label
df["CustomerID"] = df["CustomerID"].astype(str).replace("0", "Missing")

# Clean up the Description column by removing whitespaces and converting text to uppercase for uniformity
df["Description"] = df["Description"].fillna("Missing")
df["Description"] = df["Description"].str.strip().str.upper()

# Display the structural summary of the cleaned DataFrame to verify data types and non-null counts
print(df.info())


<class 'pandas.DataFrame'>
Index: 524878 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    524878 non-null  str           
 1   StockCode    524878 non-null  str           
 2   Description  524878 non-null  str           
 3   Quantity     524878 non-null  int64         
 4   InvoiceDate  524878 non-null  datetime64[us]
 5   UnitPrice    524878 non-null  float64       
 6   CustomerID   524878 non-null  str           
 7   Country      524878 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(5)
memory usage: 64.5 MB
None


In [63]:
# Create a new column calculated by multiplying Quantity and UnitPrice to get the total transactional value
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

In [64]:
df["StockCode"].unique()


<ArrowStringArray>
['85123A',  '71053', '84406B', '84029G', '84029E',  '22752',  '21730',
  '22633',  '22632',  '84879',
 ...
  '23560',  '23576',  '23562',  '23561',  '23609', '85179a',  '23617',
 '90214U', '47591b',  '23843']
Length: 3922, dtype: str

In [65]:
# Remove non product transactions such as postage or manual adjustments from the StockCode column
# This filters out rows where the StockCode is explicitly letters like POST, DOT, M, or BANK CHARGES
invalid_codes = ["POST", "DOT", "M", "BANK CHARGES", "PADS"]
df = df[~df["StockCode"].isin(invalid_codes)]

# Print the final shape of the dataset to verify how many rows and columns remain
print(df.shape)

(522715, 9)


In [66]:
# Create the formatting copy 
df_export = df.copy()

# Export the fully cleaned DataFrame to a new CSV file without saving the index row
df.to_csv(
    r"C:\Users\sanjay\OneDrive\Desktop\CS50\004.online_retail_dataset\OnlineRetail_Cleaned.csv", 
    index=False
)

print("Data export successful. Ready for import.")

Data export successful. Ready for import.
